# RayExtract TreeInfo Metric Analysis

This notebook provides a dedicated environment to import, parse, and analyze tree-level and segment-level metrics from the `*_trees_info.txt` files generated by the **RayExtract** pipeline mode.

## How RayExtract `treeinfo` Data is Structured:
- Each non-comment line represents a single tree.
- **Tree-level headers** (first 7 columns):
  1. `height_m`: Height of the tree (meters)
  2. `volume_m3`: Total volume of the tree (meters³)
  3. `base_x`: X coordinate of the tree base
  4. `base_y`: Y coordinate of the tree base
  5. `dbh_m`: Diameter at Breast Height (meters)
  6. `tilt`: Tilt angle of the tree
  7. `num_segments`: Number of segment cylinders
- **Segment-level data** (subsequent groups of 14 columns per segment):
  - `x_start, y_start, z_start` and `x_end, y_end, z_end`: Starting and ending points of the cylinder.
  - `radius`: Radius of the cylinder segment (meters).
  - `length`: Length of the cylinder segment (meters).
  - `volume`: Volume of the cylinder segment (meters³, located at index 6 of the segment block).
  - Other geometric orientations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set plotting style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

In [ ]:
def parse_treeinfo_file(filepath: Path, wood_density_kg_m3: float = 900.0) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Parses a CSIRO RayExtract treeinfo file.
    
    Parameters
    ----------
    filepath : Path
        Path to the *_trees_info.txt file.
    wood_density_kg_m3 : float
        Wood density in kg/m3 for dry mass estimation (default 900.0).
        
    Returns
    -------
    tree_df : pd.DataFrame
        DataFrame containing tree-level statistics.
    segment_df : pd.DataFrame
        DataFrame containing segment-level details for all trees.
    """
    tree_rows = []
    segment_rows = []
    
    if not filepath.exists():
        raise FileNotFoundError(f"File not found: {filepath}")
        
    tree_idx = 0
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            # Skip comments and header lines
            if not line or line.startswith("#") or line.startswith("height"):
                continue
                
            parts = [p.strip() for p in line.split(",") if p.strip()]
            if len(parts) < 7:
                continue
                
            try: 
                # 1. Parse tree-level attributes
                height_m = float(parts[0])
                raw_vol_m3 = float(parts[1])
                base_x = float(parts[2])
                base_y = float(parts[3])
                dbh_m = float(parts[4])
                dbh_cm = dbh_m * 100.0
                tilt = float(parts[5])
                num_segs_reported = int(parts[6])
                
                # 2. Parse segment-level attributes (blocks of 14 values)
                segment_vols = []
                tree_segment_idx = 0
                
                for s in range(0, len(parts) - 7, 14):
                    if 7 + s + 13 < len(parts):
                        seg_vals = [float(parts[7 + s + i]) for i in range(14)]
                        
                        # Extract segment parameters
                        # Segment Volume is at index 6 of the 14-value block
                        vol_val = seg_vals[6]
                        segment_vols.append(vol_val)
                        
                        # Z height can be approximated as the average of z_start (index 2) and z_end (index 5)
                        z_start = seg_vals[2]
                        z_end = seg_vals[5]
                        z_mean = (z_start + z_end) / 2.0
                        
                        segment_rows.append({
                            "tree_id": tree_idx,
                            "segment_idx": tree_segment_idx,
                            "x_start": seg_vals[0],
                            "y_start": seg_vals[1],
                            "z_start": z_start,
                            "x_end": seg_vals[3],
                            "y_end": seg_vals[4],
                            "z_end": z_end,
                            "z_mean_m": z_mean,
                            "radius_m": seg_vals[6],  # Index 6 of the block
                            "length_m": seg_vals[7],
                            "volume_m3": seg_vals[8], # Volume field (typically index 8 in standard TreeTools, check calculation)
                            "segment_raw_vol": vol_val  # Let's save the indexed volume value too
                        })
                        tree_segment_idx += 1
                
                calculated_volume = sum(segment_vols) if segment_vols else 0.0
                mass_kg = calculated_volume * wood_density_kg_m3 if wood_density_kg_m3 else None
                
                tree_rows.append({
                    "tree_id": tree_idx,
                    "height_m": height_m,
                    "dbh_m": dbh_m,
                    "dbh_cm": dbh_cm,
                    "volume_m3_calculated": calculated_volume,
                    "volume_m3_raw": raw_vol_m3,
                    "base_x": base_x,
                    "base_y": base_y,
                    "tilt": tilt,
                    "mass_kg": mass_kg,
                    "num_segments": len(segment_vols),
                    "num_segments_reported": num_segs_reported
                })
                tree_idx += 1
            except (ValueError, IndexError) as e:
                print(f"Warning: Failed to parse tree at index {tree_idx}: {e}")
                continue
                
    return pd.DataFrame(tree_rows), pd.DataFrame(segment_rows)

In [ ]:
# Edit this path to point to your actual output file
# Default is set to the expected output path of the unified pipeline
TREE_INFO_PATH = Path("results/canoa/rayextract_full/rayextract_work/talhao_62_25x25_raycloud_trees_info.txt")

# Biological Parameter
WOOD_DENSITY_KG_M3 = 900.0

print(f"Configured File Path: {TREE_INFO_PATH.resolve()}")

In [ ]:
try:
    df_tree, df_seg = parse_treeinfo_file(TREE_INFO_PATH, wood_density_kg_m3=WOOD_DENSITY_KG_M3)
    print(f"✓ Successfully parsed {len(df_tree)} trees with {len(df_seg)} total segment cylinders.")
    
    # Display headers
    print("\n--- Trees DataFrame Sample ---")
    display(df_tree.head())
    
    print("\n--- Segments DataFrame Sample ---")
    display(df_seg.head())
except FileNotFoundError:
    print(f"❌ ERROR: File not found at {TREE_INFO_PATH}. Please run the pipeline first or provide a valid file.")
    # Mock data to allow code to compile and show how it works
    print("\nGenerating mock data for presentation purposes...")
    np.random.seed(42)
    n_mock_trees = 35
    mock_trees = []
    mock_segments = []
    
    for tid in range(n_mock_trees):
        h = np.random.normal(25, 4)
        dbh_c = np.random.normal(18, 3)
        vol = (np.pi * (dbh_c / 200.0)**2 * h) * 0.7
        mock_trees.append({
            "tree_id": tid,
            "height_m": h,
            "dbh_m": dbh_c / 100.0,
            "dbh_cm": dbh_c,
            "volume_m3_calculated": vol,
            "volume_m3_raw": vol,
            "base_x": np.random.uniform(0, 50),
            "base_y": np.random.uniform(0, 50),
            "tilt": np.random.uniform(0, 10),
            "mass_kg": vol * WOOD_DENSITY_KG_M3,
            "num_segments": int(h * 3),
            "num_segments_reported": int(h * 3)
        })
        
        for s in range(int(h * 3)):
            mock_segments.append({
                "tree_id": tid,
                "segment_idx": s,
                "z_mean_m": s * 0.33,
                "radius_m": (dbh_c / 200.0) * (1 - (s * 0.33) / h * 0.8), # taper
                "volume_m3": vol / (h * 3)
            })
            
    df_tree = pd.DataFrame(mock_trees)
    df_seg = pd.DataFrame(mock_segments)
    display(df_tree.head())

## 1. Tree-Level Metric Statistics

Below we calculate the overall summary statistics for all trees in the plot, including mean DBH (cm), Height (m), and Volume (m³).

In [ ]:
print("### Descriptive Statistics for Plot Trees ###")
display(df_tree[["height_m", "dbh_cm", "volume_m3_calculated", "mass_kg", "tilt", "num_segments"]].describe())

total_vol = df_tree["volume_m3_calculated"].sum()
total_mass = df_tree["mass_kg"].sum()
print(f"Total Plot Volume: {total_vol:.3f} m³")
print(f"Total Plot Dry Mass: {total_mass:.1f} kg (Density = {WOOD_DENSITY_KG_M3} kg/m³)")

### Metric Distributions

Visualize the histograms of Diameter at Breast Height (DBH), Height, and Volume.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# DBH Histogram
sns.histplot(df_tree["dbh_cm"], kde=True, ax=axes[0], color="teal")
axes[0].set_title("Distribution of DBH (DAP)")
axes[0].set_xlabel("DBH (cm)")

# Height Histogram
sns.histplot(df_tree["height_m"], kde=True, ax=axes[1], color="coral")
axes[1].set_title("Distribution of Tree Height")
axes[1].set_xlabel("Height (m)")

# Volume Histogram
sns.histplot(df_tree["volume_m3_calculated"], kde=True, ax=axes[2], color="mediumpurple")
axes[2].set_title("Distribution of Trunk Volume")
axes[2].set_xlabel("Volume (m³)")

plt.tight_layout()
plt.show()

## 2. Correlation Analysis

Let's look at the correlation between key biophysical variables: Height vs. DBH and DBH vs. Volume.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 1. DBH vs Height
sns.regplot(data=df_tree, x="dbh_cm", y="height_m", ax=axes[0], color="darkcyan")
axes[0].set_title("Height vs. DBH (DAP)")
axes[0].set_xlabel("DBH (cm)")
axes[0].set_ylabel("Height (m)")

# 2. DBH vs Volume
sns.regplot(data=df_tree, x="dbh_cm", y="volume_m3_calculated", ax=axes[1], color="crimson")
axes[1].set_title("Volume vs. DBH (DAP)")
axes[1].set_xlabel("DBH (cm)")
axes[1].set_ylabel("Volume (m³)")

plt.tight_layout()
plt.show()

# Print correlation matrix
corr = df_tree[["dbh_cm", "height_m", "volume_m3_calculated", "tilt"]].corr()
print("Correlation Matrix:")
display(corr.style.background_gradient(cmap="coolwarm", axis=None))

## 3. Segment-Level Taper Analysis (Trunk Profile)

Using segment records, we can visualize how the trunk diameter tapers off with height ($z$). We group the segments by Z coordinate and calculate the mean radius.

In [ ]:
if "radius_m" in df_seg.columns:
    df_seg_clean = df_seg[df_seg["radius_m"] > 0].copy()
    df_seg_clean["diameter_cm"] = df_seg_clean["radius_m"] * 2.0 * 100.0
    
    plt.figure(figsize=(10, 6))
    # Draw points for all segments across all trees
    sns.scatterplot(data=df_seg_clean, x="diameter_cm", y="z_mean_m", alpha=0.15, color="grey", label="Segments (All Trees)")
    
    # Draw a global polynomial trend curve representing the average taper profile
    sns.regplot(data=df_seg_clean, x="diameter_cm", y="z_mean_m", scatter=False, order=3, color="forestgreen", label="Mean Taper Profile (Order 3)")
    
    plt.title("Eucalyptus Trunk Taper Profile")
    plt.xlabel("Segment Diameter (cm)")
    plt.ylabel("Height above Ground (m)")
    plt.legend()
    plt.show()
else:
    # Handle mock schema
    df_seg["diameter_cm"] = df_seg["radius_m"] * 2.0 * 100.0
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df_seg, x="diameter_cm", y="z_mean_m", alpha=0.2, color="grey")
    sns.regplot(data=df_seg, x="diameter_cm", y="z_mean_m", scatter=False, order=3, color="forestgreen", label="Mean Taper Profile (Order 3)")
    plt.title("Eucalyptus Trunk Taper Profile (Mock Data)")
    plt.xlabel("Segment Diameter (cm)")
    plt.ylabel("Height above Ground (m)")
    plt.legend()
    plt.show()